# Data Processing

In [1]:
import pandas as pd
import random

### IEMOCAP train, dev, test split

In [2]:
# Load dataset
file_path = "./iemocap.csv"
df = pd.read_csv(file_path)

df

,Dialogue_ID,Utterance_ID,Speaker,Emotion,Utterance
0,0,0,Ses01F_impro01_F,neutral,Excuse me.
1,0,1,Ses01F_impro01_M,frustration,Do you have your forms?
2,0,2,Ses01F_impro01_F,neutral,Yeah.
3,0,3,Ses01F_impro01_M,frustration,Let me see them.
4,0,4,Ses01F_impro01_F,neutral,Is there a problem?
...,...,...,...,...,...
10082,1140,4,Ses05M_script03_2_M,anger,oh! Marry you again? I wouldn't marry you agai...
10083,1140,5,Ses05M_script03_2_F,anger,Beast
10084,1140,6,Ses05M_script03_2_M,anger,You're a wicked little vampire. And I pray to...
10085,1140,7,Ses05M_script03_2_F,anger,Brute


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10087 entries, 0 to 10086
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Dialogue_ID   10087 non-null  int64 
 1   Utterance_ID  10087 non-null  int64 
 2   Speaker       10087 non-null  object
 3   Emotion       10087 non-null  object
 4   Utterance     10086 non-null  object
dtypes: int64(2), object(3)
memory usage: 394.1+ KB


In [4]:
# Group by Dialogue ID to keep utterances together
dialogue_groups = df.groupby("Dialogue_ID")

In [5]:
# Convert groups to a list of DataFrames (each representing a full dialogue)
dialogue_list = [group for _, group in dialogue_groups]

dialogue_list

[   Dialogue_ID  Utterance_ID           Speaker      Emotion  \
 0            0             0  Ses01F_impro01_F      neutral   
 1            0             1  Ses01F_impro01_M  frustration   
 2            0             2  Ses01F_impro01_F      neutral   
 3            0             3  Ses01F_impro01_M  frustration   
 4            0             4  Ses01F_impro01_F      neutral   
 5            0             5  Ses01F_impro01_M  frustration   
 6            0             6  Ses01F_impro01_F      neutral   
 7            0             7  Ses01F_impro01_F      neutral   
 8            0             8  Ses01F_impro01_M  frustration   
 9            0             9  Ses01F_impro01_F      neutral   
 
                                            Utterance  
 0                                         Excuse me.  
 1                            Do you have your forms?  
 2                                              Yeah.  
 3                                   Let me see them.  
 4            

In [6]:
# Shuffle the dialogues (not individual utterances)
random.seed(1024)
random.shuffle(dialogue_list)

In [7]:
# Compute first split (train 80%, test 20%)
total_dialogues = len(dialogue_list)
train_size = int(total_dialogues * 0.8)

train_dialogues = dialogue_list[:train_size]
test_dialogues = dialogue_list[train_size:]

In [8]:
# Compute second split (train 90%, dev 10%)
train_size_final = int(len(train_dialogues) * 0.9)

final_train_dialogues = train_dialogues[:train_size_final]
dev_dialogues = train_dialogues[train_size_final:]

In [9]:
# Function to reset Dialogue_IDs starting from 0
def reset_dialogue_ids(dialogues):
    new_dialogues = []
    for new_id, dialogue in enumerate(dialogues):  # Start at 0
        dialogue = dialogue.copy()
        dialogue["Dialogue_ID"] = new_id  # Assign new sequential ID
        new_dialogues.append(dialogue)
    return pd.concat(new_dialogues).reset_index(drop=True)


In [10]:
# Reset Dialogue_IDs for each split
train_df = reset_dialogue_ids(final_train_dialogues)
dev_df = reset_dialogue_ids(dev_dialogues)
test_df = reset_dialogue_ids(test_dialogues)

In [11]:
train_df['Utterance'] = train_df['Utterance'].fillna('')
dev_df['Utterance'] = dev_df['Utterance'].fillna('')
test_df['Utterance'] = test_df['Utterance'].fillna('')

In [12]:
# Save datasets
train_df.to_csv("../datasets/iemocap_train.csv", index=False)
dev_df.to_csv("../datasets/iemocap_dev.csv", index=False)
test_df.to_csv("../datasets/iemocap_test.csv", index=False)

In [13]:
train = pd.read_csv("../datasets/iemocap_train.csv")
dev = pd.read_csv("../datasets/iemocap_dev.csv")
test = pd.read_csv("../datasets/iemocap_test.csv")

train_len = train["Dialogue_ID"].max()
dev_len = dev["Dialogue_ID"].max()
test_len = test["Dialogue_ID"].max()

print(train_len)
print(dev_len)
print(test_len)

total = train_len + dev_len + test_len

print(round(train_len / total, 2))
print(round(dev_len / total, 2))
print(round(test_len / total, 2))

819
91
228
0.72
0.08
0.2


In [14]:
import json
from collections import defaultdict

In [15]:
# Load CSV
df = pd.read_csv("../datasets/iemocap_train.csv")

# Group by Dialogue_ID
dialogues = defaultdict(list)

for dial_id, group in df.groupby("Dialogue_ID"):
    utterances = []
    for _, row in group.iterrows():
        utt_id = f"dialogue_{dial_id}_c01_u{int(row['Utterance_ID']):03d}"
        utterance_entry = {
            "utterance_id": utt_id,
            "speakers": [row["Speaker"]],
            "transcript": row["Utterance"]
        }
        utterances.append(utterance_entry)
    
    scene = {
        "scene_id": f"dialogue_{dial_id}_c01",
        "scenes_name": "N/A",
        "utterances": utterances
    }
    dialogues[f"dialogue_{dial_id}"].append(scene)

# Save to JSON
with open("../emotionflow/transcripts/iemocap_transcript.json", "w") as f:
    json.dump(dialogues, f, indent=2)